<a href="https://colab.research.google.com/github/Future-Analyst/Tensorflow-Exercises-/blob/main/03_CNN_exercise_usign_cifar10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## base line cnn model

In [21]:
# Imports for self-containment
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten, MaxPool2D, Conv2D
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical

# Re-load the CIFAR-10 dataset to get fresh integer labels
(x_train_full, y_train_full), (x_test_full, y_test_full) = cifar10.load_data()

# Limit the dataset to 1000 samples for consistency with previous steps
x_train_model = x_train_full[:1000]
y_train_model = y_train_full[:1000]

x_test_model = x_test_full[:1000]
y_test_model = y_test_full[:1000]

# Normalize image data
x_train_model = x_train_model / 255.0
x_test_model = x_test_model / 255.0

# One-hot encode the labels correctly for CategoricalCrossentropy
# This converts labels from (num_samples, 1) integer format to (num_samples, num_classes) binary format
y_train_model = to_categorical(y_train_model, num_classes=10)
y_test_model = to_categorical(y_test_model, num_classes=10)



In [23]:
# baseline model
model_0 = Sequential([
    Conv2D(32, 3, activation='relu', input_shape=(32,32,3)),
    MaxPool2D(2),
    Conv2D(32, 3, activation='relu'),
    Conv2D(32, 3, activation='relu'),
    MaxPool2D(2),
    Flatten(),
    Dense(10, activation='softmax')
])

model_0.compile(
    loss=CategoricalCrossentropy(),
    optimizer=SGD(learning_rate=0.01),
    metrics=['accuracy']
)

model_0.fit(
    x_train_model,
    y_train_model,
    epochs=5,
    validation_data=(x_test_model, y_test_model)
)


Epoch 1/5


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 51ms/step - accuracy: 0.1090 - loss: 2.3066 - val_accuracy: 0.1130 - val_loss: 2.3012
Epoch 2/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 42ms/step - accuracy: 0.1110 - loss: 2.3008 - val_accuracy: 0.1260 - val_loss: 2.2986
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.1300 - loss: 2.2979 - val_accuracy: 0.1300 - val_loss: 2.2983
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 1s 43ms/step - accuracy: 0.1440 - loss: 2.2956 - val_accuracy: 0.1460 - val_loss: 2.2962
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 50ms/step - accuracy: 0.1540 - loss: 2.2929 - val_accuracy: 0.1560 - val_loss: 2.2949


In [25]:
# check the layers in our baseline model
model_0.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_9 (Conv2D)               │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_10 (Conv2D)              │ (None, 13, 13, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_11 (Conv2D)              │ (None, 11, 11, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 5, 5, 32)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 800)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 10)             │         8,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 27,404 (107.05 KB)

 Trainable params: 27,402 (107.04 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [26]:
# Evaluate the baseline model
model_0.evaluate(x_test_model, y_test_model)

32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.1560 - loss: 2.2949


[2.294865131378174, 0.15600000321865082]

In [31]:
# Build a second model
import tensorflow as tf
tf.random.set_seed(42)

# Create a model to improve the baseline model

model_1 = Sequential([
    # Convolutional Base for feature extraction
    Conv2D(32, 3, activation='elu', input_shape=(32,32,3)),
    MaxPool2D(2),
    Conv2D(64, 3, activation='elu'), # Increased filters for more complex features
    MaxPool2D(2),
    Conv2D(128, 3, activation='elu'), # Added another convolutional layer

    # Flatten the output of the convolutional layers to feed into Dense layers
    Flatten(),

    # Dense layers for classification
    Dense(100, activation='elu'),
    Dense(100, activation='elu'),
    Dense(10, activation='softmax') # Output layer for 10 classes
])

# Compile the model
model_1.compile(loss = CategoricalCrossentropy(),
                optimizer=SGD(learning_rate=0.01),
                metrics=['accuracy'])

# Fit the model
model_1.fit(
    x_train_model,
    y_train_model,
    epochs=20,
    validation_data=(x_test_model, y_test_model),
    steps_per_epoch=len(x_train_model) // 32
)

Epoch 1/20


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 68ms/step - accuracy: 0.1140 - loss: 2.3043 - val_accuracy: 0.1410 - val_loss: 2.2859
Epoch 2/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 60ms/step - accuracy: 0.1710 - loss: 2.2688 - val_accuracy: 0.1940 - val_loss: 2.2605
Epoch 3/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - accuracy: 0.2190 - loss: 2.2360 - val_accuracy: 0.2140 - val_loss: 2.2304
Epoch 4/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 57ms/step - accuracy: 0.2560 - loss: 2.1975 - val_accuracy: 0.2380 - val_loss: 2.1932
Epoch 5/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 3s 90ms/step - accuracy: 0.2890 - loss: 2.1509 - val_accuracy: 0.2520 - val_loss: 2.1492
Epoch 6/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step - accuracy: 0.3120 - loss: 2.0995 - val_accuracy: 0.2590 - val_loss: 2.1054
Epoch 7/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.3220 - loss: 2.0508 - val_accuracy: 0.2620 - val_loss: 2.0681
Epoch 8/20
31/31 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 0.3200 - loss: 2.0085 - val_accuracy: 0.2700 - val_loss: 2.

In [32]:
# Build a third model by just increasing the numer epochs to 50
import tensorflow as tf
tf.random.set_seed(42)

# Create a model to improve the baseline model

model_1 = Sequential([
    # Convolutional Base for feature extraction
    Conv2D(64, 3, activation='elu', input_shape=(32,32,3)),
    MaxPool2D(2),
    Conv2D(78, 3, activation='elu'), # Increased filters for more complex features
    MaxPool2D(2),
    Conv2D(158, 3, activation='elu'), # Added another convolutional layer

    # Flatten the output of the convolutional layers to feed into Dense layers
    Flatten(),

    # Dense layers for classification
    Dense(150, activation='elu'),
    Dense(150, activation='elu'),
    Dense(150, activation='elu'),
    Dense(150, activation='elu'),
    Dense(10, activation='softmax') # Output layer for 10 classes
])

# Compile the model
model_1.compile(loss = CategoricalCrossentropy(),
                optimizer=SGD(learning_rate=0.01),
                metrics=['accuracy'])

# Fit the model
model_1.fit(
    x_train_model,
    y_train_model,
    epochs=50,
    validation_data=(x_test_model, y_test_model),
    steps_per_epoch=len(x_train_model) // 62
)

Epoch 1/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 208ms/step - accuracy: 0.1070 - loss: 2.3110 - val_accuracy: 0.0880 - val_loss: 2.3045
Epoch 2/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 4s 246ms/step - accuracy: 0.1040 - loss: 2.2927 - val_accuracy: 0.0900 - val_loss: 2.2929
Epoch 3/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 180ms/step - accuracy: 0.1430 - loss: 2.2778 - val_accuracy: 0.1090 - val_loss: 2.2809
Epoch 4/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 180ms/step - accuracy: 0.1800 - loss: 2.2623 - val_accuracy: 0.1290 - val_loss: 2.2673
Epoch 5/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 181ms/step - accuracy: 0.1990 - loss: 2.2449 - val_accuracy: 0.1500 - val_loss: 2.2514
Epoch 6/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 5s 180ms/step - accuracy: 0.2310 - loss: 2.2246 - val_accuracy: 0.1730 - val_loss: 2.2321
Epoch 7/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 180ms/step - accuracy: 0.2490 - loss: 2.2004 - val_accuracy: 0.1930 - val_loss: 2.2088
Epoch 8/50
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 179ms/step - accuracy: 0.2550 - loss: 2.1717 - val_accuracy: 0.

In [ ]:
model = models.Sequential([
    layers.Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64, (3,3), activation='relu'),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    layers.Dense(10, activation='softmax')  # CIFAR-10 has 10 classes
])